# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Evidence Notebook 07 - Evaluation Protocol and Core Results

Examiner-facing recovery notebook for frozen Step 4 and Step 7 evidence. It is not an original development notebook and does not rerun retrieval or official scoring.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display, Markdown

CC = Path(r"${THESIS_CONTROL_ROOT}")
WS = Path(r"${THESIS_WORKSPACE}")
SUMMARY_PATH = Path(r"${THESIS_CONTROL_ROOT}\Evidence_Recovery_and_Thesis_Assets") / "06_Validation" / "Notebook_Summaries" / "Evidence_Notebook_07_summary.json"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

print("Evidence notebook mode: frozen-artifact verification only")
print("Correction Control:", CC)
print("Writing Workspace:", WS)


Evidence notebook mode: frozen-artifact verification only
Correction Control: ${THESIS_CONTROL_ROOT}
Writing Workspace: ${THESIS_WORKSPACE}


## Authoritative frozen inputs and official qrel identity

The notebook verifies the existing no-answer population audit, frozen qrels, and final core-result tables.

In [2]:
audit = CC / "No_Answer_Audit"
core = CC / "Final_Core_Evaluation"
sources = {
    "No-answer split summary": audit / "NO_ANSWER_SPLIT_SUMMARY.csv",
    "Core all-splits results": core / "CORE_RESULTS_ALL_SPLITS.csv",
    "Train answerable-only qrels": audit / "qrels_train_answerable_only.gold",
    "Dev answerable-only qrels": audit / "qrels_dev_answerable_only.gold",
    "Test answerable-only qrels": audit / "qrels_test_answerable_only.gold",
}
source_table = pd.DataFrame([{"Source": k, "Path": str(v), "SHA256": sha256(v)} for k,v in sources.items()])
display(source_table)
expected_qrel_hashes = {
    "Train answerable-only qrels":"FD8D351896507C33021EB8B1F5BAE403F09105495E51D7ECDA4AF44B19F93176",
    "Dev answerable-only qrels":"4D372EEF4655E27662F1F017557B01F003C6A16C57E16074621F1E227E31EAF6",
    "Test answerable-only qrels":"E799BEC96EA33B7ECDFC0B8285B85B4F16521B6C2FDD7DD02AA7AFE0F608872B",
}
for name, expected in expected_qrel_hashes.items(): assert sha256(sources[name]) == expected
print("Answerable-only qrel identities: PASS")

,Source,Path,SHA256
0,No-answer split summary,${THESIS_CODEX_ROOT}\Master Thesis - ...,DBA65E5EF668DA54A924BF535C42975EC2A9939929C123...
1,Core all-splits results,${THESIS_CODEX_ROOT}\Master Thesis - ...,F3D95EA6DD7EAAA3C4F30CDACAAC3446BD084566552CC5...
2,Train answerable-only qrels,${THESIS_CODEX_ROOT}\Master Thesis - ...,FD8D351896507C33021EB8B1F5BAE403F09105495E51D7...
3,Dev answerable-only qrels,${THESIS_CODEX_ROOT}\Master Thesis - ...,4D372EEF4655E27662F1F017557B01F003C6A16C57E160...
4,Test answerable-only qrels,${THESIS_CODEX_ROOT}\Master Thesis - ...,E799BEC96EA33B7ECDFC0B8285B85B4F16521B6C2FDD7D...


Answerable-only qrel identities: PASS


## Gold no-answer encoding and frozen evaluation populations

Gold no-answer is encoded by a relevant judgment on `Passage_ID = -1`. Qid 504 is present in the official test questions but absent from published test qrels; it is outside both scoring populations.

In [3]:
split_summary = pd.read_csv(sources["No-answer split summary"])
cols = ["Split","Official_Question_Count","Judged_QID_Count","Gold_No_Answer_Count","Gold_Answerable_Count","Qrel_Absent_Count"]
display(split_summary[cols])
expected = {"train":(174,174,26,148,0), "dev":(25,25,4,21,0), "test":(52,51,7,44,1)}
for _, r in split_summary.iterrows():
    got = tuple(int(r[c]) for c in cols[1:])
    assert got == expected[r.Split]
print("Official judged and answerable-only populations: PASS")

,Split,Official_Question_Count,Judged_QID_Count,Gold_No_Answer_Count,Gold_Answerable_Count,Qrel_Absent_Count
0,train,174,174,26,148,0
1,dev,25,25,4,21,0
2,test,52,51,7,44,1


Official judged and answerable-only populations: PASS


## System no-answer behavior and evaluation protocol

BM25 and Dense E5 always return normal passages. Fixed and tuned RALSR emit `-1` only when root-aware candidate generation returns zero candidates. No system uses a tuned no-answer threshold. Official judged results are primary; answerable-only results are a secondary analysis.

In [4]:
behavior = pd.DataFrame([
    ("BM25","No abstention","Normal ranked passages"),
    ("Dense E5","No abstention","Normal ranked passages"),
    ("Fixed RALSR","Structural only","-1 iff candidate_count = 0"),
    ("Tuned RALSR","Structural only","Same candidate set and abstentions as fixed RALSR"),
], columns=["System","No-answer mechanism","Frozen behavior"])
display(behavior)

,System,No-answer mechanism,Frozen behavior
0,BM25,No abstention,Normal ranked passages
1,Dense E5,No abstention,Normal ranked passages
2,Fixed RALSR,Structural only,-1 iff candidate_count = 0
3,Tuned RALSR,Structural only,Same candidate set and abstentions as fixed RALSR


## Frozen core runs and final metrics

Metrics are loaded from the Step-7 result freeze. Tuned-RALSR training results are optimization-set results, not independent evidence.

In [5]:
results = pd.read_csv(sources["Core all-splits results"])
assert set(results.System) == {"BM25","Dense E5","Fixed RALSR","Tuned RALSR"}
assert set(results.Population) == {"Official judged","Answerable-only"}
display(results.pivot_table(index=["Split","System"], columns="Population", values=["MAP_at_10","MRR_at_10"]))
test = results[results.Split.eq("test")][["System","Population","QID_Count","MAP_at_10","MRR_at_10"]]
display(test)

MAP_at_10                       MRR_at_10  \
Population        Answerable-only Official judged Answerable-only   
Split System                                                        
dev   BM25               0.128084        0.107590        0.289286   
      Dense E5           0.189828        0.159456        0.357332   
      Fixed RALSR        0.080215        0.067381        0.150794   
      Tuned RALSR        0.080074        0.067262        0.130952   
test  BM25               0.085027        0.073357        0.201948   
      Dense E5           0.115360        0.099527        0.335606   
      Fixed RALSR        0.049721        0.042897        0.156656   
      Tuned RALSR        0.047341        0.040844        0.157477   
train BM25               0.137172        0.116675        0.233770   
      Dense E5           0.236648        0.201287        0.386653   
      Fixed RALSR        0.105049        0.100846        0.168439   
      Tuned RALSR        0.112085        0.106831        0.189321   

                                   
Population        Official judged  
Split System                       
dev   BM25               0.243000  
      Dense E5           0.300159  
      Fixed RALSR        0.126667  
      Tuned RALSR        0.110000  
test  BM25               0.174230  
      Dense E5           0.289542  
      Fixed RALSR        0.135154  
      Tuned RALSR        0.135862  
train BM25               0.198839  
      Dense E5           0.328877  
      Fixed RALSR        0.154764  
      Tuned RALSR        0.172526

,System,Population,QID_Count,MAP_at_10,MRR_at_10
2,BM25,Official judged,51,0.073357,0.174230
5,Dense E5,Official judged,51,0.099527,0.289542
8,Fixed RALSR,Official judged,51,0.042897,0.135154
11,Tuned RALSR,Official judged,51,0.040844,0.135862
14,BM25,Answerable-only,44,0.085027,0.201948
17,Dense E5,Answerable-only,44,0.115360,0.335606
20,Fixed RALSR,Answerable-only,44,0.049721,0.156656
23,Tuned RALSR,Answerable-only,44,0.047341,0.157477


## Reproducibility evidence

The preserved Step-7 repeated-scoring and independent cross-check records are displayed without introducing a new evaluator.

In [6]:
repro = json.loads((core / "CORE_SCORING_REPRODUCIBILITY.json").read_text(encoding="utf-8"))
cross = json.loads((core / "CORE_EVALUATOR_CROSSCHECK.json").read_text(encoding="utf-8"))
display(repro)
display(cross)
summary = {
    "train_judged":174,"dev_judged":25,"test_judged":51,
    "train_answerable":148,"dev_answerable":21,"test_answerable":44,
    "qid_504_qrel_absent":True,
    "dense_test_official_map":float(results.query("Split=='test' and System=='Dense E5' and Population=='Official judged'").MAP_at_10.iloc[0]),
    "fixed_test_official_map":float(results.query("Split=='test' and System=='Fixed RALSR' and Population=='Official judged'").MAP_at_10.iloc[0]),
    "substantive_mismatches":0,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Notebook summary written:", SUMMARY_PATH)
print("Substantive mismatch count: 0")

{'checkpoint_results': {'checkpoint_Fixed_RALSR_train': {'map_cut_10': 0.1008461549182334,
   'recip_rank': 0.1547641853676336,
   'metrics_path': 'C:\\Users\\User\\Documents\\Codex\\Master Thesis - Quranic Passage Retrieval - Correction Control\\Final_Core_Evaluation\\Evaluator_Logs\\checkpoint_Fixed_RALSR_train.metrics.tsv',
   'expected': {'map_cut_10': 0.100846154918, 'recip_rank': 0.154764185368},
   'max_delta': 3.664013537019173e-13,
   'passed': True},
  'checkpoint_Fixed_RALSR_dev': {'map_cut_10': 0.0673809523809523,
   'recip_rank': 0.1266666666666666,
   'metrics_path': 'C:\\Users\\User\\Documents\\Codex\\Master Thesis - Quranic Passage Retrieval - Correction Control\\Final_Core_Evaluation\\Evaluator_Logs\\checkpoint_Fixed_RALSR_dev.metrics.tsv',
   'expected': {'map_cut_10': 0.067380952381, 'recip_rank': 0.126666666667},
   'max_delta': 3.333999742949345e-13,
   'passed': True},
  'checkpoint_Tuned_RALSR_train': {'map_cut_10': 0.1068313380957059,
   'recip_rank': 0.17252554

{'preserved_independent_evaluator': 'C:\\Users\\User\\Documents\\Codex\\Master Thesis - Quranic Passage Retrieval - Correction Control\\Official_QuranQA_TaskA_Evaluation\\02_Evaluation_Pipeline\\quranqa_taskA_independent_eval.py',
 'conditions_checked': 24,
 'maximum_absolute_difference': 5.689893001203927e-16,
 'tolerance': 1e-12,
 'passed': True,
 'checks': [{'condition': 'Official_judged_BM25_train',
   'official_scorer_map_cut_10': 0.116674984950847,
   'independent_map_cut_10': 0.11667498495084702,
   'map_delta': 2.7755575615628914e-17,
   'official_scorer_recip_rank': 0.198839171683999,
   'independent_recip_rank': 0.19883917168399928,
   'mrr_delta': 2.7755575615628914e-16,
   'passed': True},
  {'condition': 'Official_judged_BM25_dev',
   'official_scorer_map_cut_10': 0.107590476190476,
   'independent_map_cut_10': 0.1075904761904762,
   'map_delta': 1.942890293094024e-16,
   'official_scorer_recip_rank': 0.243,
   'independent_recip_rank': 0.243,
   'mrr_delta': 0.0,
   'pass

Notebook summary written: ${THESIS_CONTROL_ROOT}\Evidence_Recovery_and_Thesis_Assets\06_Validation\Notebook_Summaries\Evidence_Notebook_07_summary.json
Substantive mismatch count: 0


## Final interpretation and thesis references

The organizer-judged evaluation remains primary, while the answerable-only population isolates passage ranking from explicit no-answer handling. Use the Step-12 no-answer table and existing Step-10 final system comparison assets in the thesis.